# 第 06 章：基础设施 (Observability & Persistence)

根据讲义，本章你将掌握以下核心能力：
- 🎯 **LangSmith 全生命周期追踪**: 建立 `LANGCHAIN_TRACING_V2` 环境，将推理过程可视化。
- 🔍 **LangFuse 开源平替**: 理解零侵入切换的回调机制，掌握私有化部署方案。
- 💾 **Checkpointer 持久化**: 引入 `MemorySaver`，理解 Agent 如何利用"存储快照"实现长效记忆。
- 🔀 **Thread ID 隔离**: 掌握多租户环境下，如何通过线程 ID 隔离不同用户的对话上下文。

## 1. 环境准备 (Environment Setup)

加载 `.env` 环境变量并初始化模型。本章的所有实验都依赖这个 Cell。

In [4]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import create_react_agent

load_dotenv()

llm = ChatOpenAI(
    model="deepseek-chat", 
    api_key=os.getenv("DEEPSEEK_API_KEY"), 
    base_url="https://api.deepseek.com"
)

print(f"模型加载成功：{llm.__class__.__name__}")

模型加载成功：ChatOpenAI


---
## 2. 场景演练 A：可观测性 —— Agent 的显微镜

### 2.1 什么是 Tracing（追踪）？

> 如果说 `print` 是在墙上贴小广告，那 **LangSmith** 就是全城监控摄像头。
> 它能记录下 Prompt 被翻译成什么 JSON、工具返回了什么原始报错、甚至每一秒产生了多少 Token 消耗。

在没有追踪工具的情况下，调试一个多步骤 Agent 几乎是盲人摸象：
- 🤔 模型到底收到了什么 Prompt？
- 🤔 工具被调用了几次？每次传了什么参数？
- 🤔 哪个步骤最慢？Token 消耗在哪里？

追踪系统就是为了回答这些问题而存在的。

### 2.2 LangSmith 环境激活

LangSmith 是 LangChain 官方提供的云端追踪平台。它的配置方式极其简单——你只需要在 `.env` 文件中加入 3 行环境变量，LangChain 框架就会**自动拦截并上报所有流量**：

```bash
# 在你项目根目录的 .env 文件中添加：
LANGCHAIN_TRACING_V2=true
LANGCHAIN_API_KEY=ls__...  # 从 smith.langchain.com 获取
LANGCHAIN_PROJECT="Logbook-Lab"
```

> **📝 获取步骤**：访问 https://smith.langchain.com → 注册/登录 → Settings → API Keys → Create API Key

下方代码帮你检测当前环境变量是否已正确配置：

In [5]:
# 诊断：检查 LangSmith 相关环境变量是否已配置
tracing_v2 = os.getenv('LANGCHAIN_TRACING_V2')
api_key = os.getenv('LANGCHAIN_API_KEY')
project = os.getenv('LANGCHAIN_PROJECT')

print("=" * 50)
print("🔍 LangSmith 环境诊断报告")
print("=" * 50)
print(f"LANGCHAIN_TRACING_V2 : {tracing_v2 or '❌ 未配置'}")
print(f"LANGCHAIN_API_KEY    : {'✅ 已配置 (' + api_key[:8] + '...)' if api_key else '❌ 未配置'}")
print(f"LANGCHAIN_PROJECT    : {project or '⚠️  未配置 (将使用默认项目)'}")
print("=" * 50)

if tracing_v2 and api_key:
    print("\n✅ LangSmith 已激活！运行下方代码后，到 smith.langchain.com 查看追踪轨迹。")
else:
    print("\n⚠️  LangSmith 未激活。你仍然可以完成本章的所有持久化实验。")
    print("   如需启用，请编辑项目根目录的 .env 文件后重启 Kernel。")

🔍 LangSmith 环境诊断报告
LANGCHAIN_TRACING_V2 : true
LANGCHAIN_API_KEY    : ✅ 已配置 (lsv2_pt_...)
LANGCHAIN_PROJECT    : Logbook-Lab

✅ LangSmith 已激活！运行下方代码后，到 smith.langchain.com 查看追踪轨迹。


### 2.3 实验：亲手观察一次追踪轨迹

即使你没有配置 LangSmith，下方代码也能正常运行。
但如果你**已配置**，运行后可以到 [smith.langchain.com](https://smith.langchain.com) 的项目面板中立刻看到这次调用的完整追踪链路：

```
链路示意：
ChatOpenAI.invoke()
  ├─ 输入 Prompt (原始 JSON)
  ├─ LLM 推理 (耗时 / Token 消耗)
  └─ 输出 AIMessage
```

**📝 观测要点：**
1. **Latency (时延)**：哪一个步骤最慢？
2. **Inputs/Outputs**：模型接收到的原始 Prompt 是什么形状？
3. **Token Usage**：这次调用消耗了多少 Token？

In [6]:
# 发起一次简单调用，观察追踪效果
response = llm.invoke([HumanMessage(content="请用一句话解释什么是 Checkpoint。")])

print(f"模型回复: {response.content}")
print(f"\n--- 元数据 ---")
print(f"模型: {response.response_metadata.get('model_name', 'N/A')}")
print(f"Token 消耗: {response.response_metadata.get('token_usage', 'N/A')}")

if os.getenv('LANGCHAIN_TRACING_V2'):
    print(f"\n🔗 前往 https://smith.langchain.com 查看此次追踪的完整链路！")

模型回复: Checkpoint 是训练过程中保存的模型状态快照，用于恢复训练或进行推理。

--- 元数据 ---
模型: deepseek-chat
Token 消耗: {'completion_tokens': 20, 'prompt_tokens': 12, 'total_tokens': 32, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 12}

🔗 前往 https://smith.langchain.com 查看此次追踪的完整链路！


### 2.4 工程决策：LangFuse —— 开源平替方案

> **[Production Decision]**：如果你所在的组织对数据隐私有极高要求，或者希望规避 LangSmith 的付费成本，
> **LangFuse** 是目前的最佳开源方案。

| 维度 | LangSmith | LangFuse |
|------|-----------|----------|
| 部署 | ☁️ SaaS 云服务 | 🏠 支持 Docker 私有化部署 |
| 成本 | 免费额度 + 付费 | 完全开源免费 |
| 数据隐私 | 数据上报至 LangChain 服务器 | 数据留在你自己的服务器 |
| 接入方式 | 环境变量（零代码） | Callback Handler（约 3 行代码） |

**切换方法**：LangChain 提供了标准的回调处理器（Callback Handler），只需将 `on_end` 等钩子对接到 LangFuse 服务器即可：

```python
# LangFuse 接入示意（约 3 行代码）
from langfuse.callback import CallbackHandler

langfuse_handler = CallbackHandler(
    public_key="pk-lf-...",
    secret_key="sk-lf-...",
    host="http://localhost:3000"  # 私有化部署地址
)

# 在调用时传入回调，即可将追踪数据发送到 LangFuse
llm.invoke("你好", config={"callbacks": [langfuse_handler]})
```

> **📝 总结**：可观测性工具的选择取决于你的部署环境和隐私需求。两者的追踪能力几乎等价，
> 核心区别在于数据存放位置。本教程后续的实验代码对两者都兼容。

---
## 3. 场景演练 B：持久化读档 (MemorySaver)

### 3.1 为什么需要持久化？

> 大语言模型本身是**无状态**的——它就像一个有间歇性失忆症的天才，每次对话都是一张白纸。

为了让 Agent 跨调用保持记忆，我们需要引入一个外部“存档点”——`Checkpointer`。

> **💡 原理直觉：单机游戏的 Save & Load**
>
> Agent 运行完一个节点，系统就会自动把当前的 `State`（包括对话历史、变量）生成一个二进制快照存进数据库。
> 下次你带着同一个卡槽（`thread_id`）回来，Agent 瞬间就能读档恢复。

### 3.2 核心概念：什么是 Config？

在 LangChain/LangGraph 中，调用 Agent 时除了输入 `input`，通常还会传入一个 `config` 字典：

```python
config = {
    "configurable": {
        "thread_id": "user_001"  # 👈 这是对话的“卡槽 ID”
    }
}
```

**为什么要这么设计？**
1. **侧信道传输**：`thread_id` 是给系统架构看的（用于找回数据库里的存档），不应该混合在给 AI 看的对话内容里。
2. **多租户隔离**：通过更换 `thread_id`，同一个 Agent 实例可以同时为成千上万个用户提供独立的、互不干扰的对话空间。

下面我们亲手验证这套机制。

### 3.3 构建带记忆的 Agent

In [ ]:
# 1. 初始化持久化层（内存级存档器，生产环境可用 Redis/Postgres 替代）
checkpointer = MemorySaver()

# 2. 绑定 Checkpointer
# 注意：我们不传 tools，让它作为一个纯聊天 Agent 演示记忆
agent_executor = create_react_agent(llm, tools=[], checkpointer=checkpointer)

print("带有'记忆读写头'的 Agent 已就绪。")

### 🧪 实验 1：建立存档 (thread_id: 'alice')

我们告诉 Agent 一些个人爱好信息，建立第一个“存档点”。

In [ ]:
# 定义 Alice 的专属卡槽
alice_config = {"configurable": {"thread_id": "alice_session_001"}}

print("--- Alice 发起对话 ---")
input_1 = {"messages": [HumanMessage(content="你好，我叫 Alice，我超级喜欢喝冰美式咖啡。")]}
res1 = agent_executor.invoke(input_1, config=alice_config)

print(f"Agent 对 Alice 的回复: {res1['messages'][-1].content}")

### 🧪 实验 2：跨调用读档

只要我们传入相同的 `alice_config`，Agent 就能从 MemorySaver 中找回之前的对话状态。

> **🤔 思考**：这里我们发起了一次**全新的 `.invoke()` 调用**，但 Agent 居然还记得 Alice 的名字和爱好。
> 秘密就在于 `checkpointer` 在幕后自动执行了 Load 操作。

In [ ]:
print("--- Alice 再次提问 ---")
input_2 = {"messages": [HumanMessage(content="还记得我喜欢喝什么吗？")]}
res2 = agent_executor.invoke(input_2, config=alice_config)

print(f"Agent 的记忆回复: {res2['messages'][-1].content}")

### 🧪 实验 3：ID 隔离验证 (thread_id: 'bob')

如果我们换成 Bob 的卡槽，Agent 应该**完全不认识 Alice**。
这就是 `thread_id` 的物理隔离防线——不同的 ID 对应完全独立的记忆空间。

In [ ]:
bob_config = {"configurable": {"thread_id": "bob_session_999"}}

print("--- Bob 乱入对话 ---")
input_3 = {"messages": [HumanMessage(content="你好，你知道我喜欢喝什么吗？")]}
res3 = agent_executor.invoke(input_3, config=bob_config)

print(f"Agent 对 Bob 的回复: {res3['messages'][-1].content}")

### 🧪 实验 4：失忆测试 —— 不传 Config 会怎样？

如果我们创建一个**不带 checkpointer** 的 Agent，即使是同一轮对话，跨调用也会彻底失忆。

In [ ]:
# 创建一个没有记忆的 Agent（不传 checkpointer）
amnesia_agent = create_react_agent(llm, tools=[])

print("--- 失忆 Agent：第 1 轮 ---")
res_a = amnesia_agent.invoke({"messages": [HumanMessage(content="我叫张三，我喜欢吃火锅。")]})
print(f"回复: {res_a['messages'][-1].content}")

print("\n--- 失忆 Agent：第 2 轮（新调用） ---")
res_b = amnesia_agent.invoke({"messages": [HumanMessage(content="还记得我叫什么吗？")]})
print(f"回复: {res_b['messages'][-1].content}")

print("\n📝 可以看到：不带 checkpointer 的 Agent 在跨调用时完全失忆。")

---
## 4. 深度解剖：快照里到底存了什么？

让我们直接打印出 `MemorySaver` 内部的存档数据，看看 `thread_id` 对应的快照结构。

In [ ]:
# 获取 Alice 线程的最新存档（使用 get_tuple 获取完整快照）
checkpoint_tuple = checkpointer.get_tuple(alice_config)

if checkpoint_tuple:
    checkpoint = checkpoint_tuple.checkpoint
    metadata = checkpoint_tuple.metadata
    
    print("=" * 50)
    print("🧠 Alice 的大脑快照数据")
    print("=" * 50)
    
    messages = checkpoint.get('channel_values', {}).get('messages', [])
    print(f"对话轮次（消息总数）: {len(messages)}")
    print(f"Checkpoint ID: {checkpoint_tuple.config['configurable'].get('checkpoint_id', 'N/A')}")
    
    print("\n--- 完整对话历史 ---")
    for i, msg in enumerate(messages):
        role = '🧑 Human' if msg.type == 'human' else '🤖 AI'
        content_preview = msg.content[:80] + ('...' if len(msg.content) > 80 else '')
        print(f"  [{i+1}] {role}: {content_preview}")
else:
    print("❌ 未找到 Alice 的存档，请先运行上方的实验 1 和实验 2。")

---
## 5. 本章小结

| 知识点 | 你学到了什么 |
|--------|-------------|
| **LangSmith** | 通过 3 行环境变量即可激活全链路追踪，在 Web 面板中可视化 Prompt / Token / 延迟 |
| **LangFuse** | 开源平替方案，约 3 行代码切换，支持私有化部署，数据不出库 |
| **MemorySaver** | 内存级 Checkpointer，生产环境可替换为 Redis / PostgreSQL |
| **thread_id** | 多租户隔离的物理防线，不同 ID = 完全独立的记忆空间 |
| **Config 侧信道** | 架构信息（如 thread_id）通过 config 透传，不污染业务 Prompt |

> **🚀 下一步**：在第 07 章中，我们将手写 `StateGraph`，彻底告别 `create_react_agent` 的黑盒，
> 亲手构建图的每一条边和每一个节点。Checkpointer 将在那里展现它的全部威力。